# TrackSense retrain-v2 (8-class) — Colab / GPU fine-tune

Closes the `whole_nuts` / `bowl` / plastic-`cutlery` gaps by fine-tuning the **existing** 8-class checkpoint on the **merged** dataset (existing multiscene + new weak-class data).

**Guardrails baked in:**
- Frozen schema — `0 nut_butter_jar 1 whole_nuts 2 hand 3 cutlery 4 chopping_board 5 plate 6 bowl 7 bread` (bread=7, no counter).
- Trains on the **merged** set only (never the new data alone) — catastrophic-forgetting guard.
- Fine-tunes **from** `tracksense_8class_multiscene_best.pt` and saves to a **NEW** file `tracksense_8class_retrain_v2.pt`. The original is read-only and never overwritten.
- Do NOT train on the CPU box — run this on Colab with a GPU runtime (Runtime ▸ Change runtime type ▸ GPU).

**You must upload to Colab first (e.g. via Google Drive):**
1. The re-sourced existing set `training_8class_multiscene/` (YOLO layout: train/valid/test → images+labels).
2. The isolated `data/retrain_8class/` produced locally by `ml/prepare_retrain_8class.py`.
3. The scripts `ml/merge_retrain_8class.py` and `ml/validate_retrain_8class.py`.
4. The existing checkpoint `tracksense_8class_multiscene_best.pt`.


In [ ]:
# 0. GPU check + install
!nvidia-smi -L || echo 'WARNING: no GPU — set Runtime > Change runtime type > GPU'
!pip -q install 'ultralytics>=8.3,<9' >/dev/null
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA GPU — do not train on CPU.'

In [ ]:
# 1. Mount Drive and set paths — EDIT these to where you uploaded things.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE       = Path('/content/drive/MyDrive/tracksense_retrain_v2')  # <-- edit
EXISTING   = BASE / 'training_8class_multiscene'   # re-sourced existing set
RETRAIN    = BASE / 'retrain_8class'               # isolated new set (from step 3)
SCRIPTS    = BASE / 'ml'                            # merge + validate scripts
SRC_CKPT   = BASE / 'tracksense_8class_multiscene_best.pt'  # existing checkpoint (READ-ONLY)
MERGED     = Path('/content/data/merged_retrain_v2')
OUT_CKPT   = BASE / 'checkpoints_retrain_v2' / 'tracksense_8class_retrain_v2.pt'  # NEW file

for p in [EXISTING, RETRAIN, SCRIPTS / 'merge_retrain_8class.py',
          SCRIPTS / 'validate_retrain_8class.py', SRC_CKPT]:
    assert p.exists(), f'MISSING: {p}'
print('all inputs present')

In [ ]:
# 2. Step 4 — MERGE existing + retrain into merged_retrain_v2 (+ class balance report)
import subprocess
rc = subprocess.run(['python', str(SCRIPTS / 'merge_retrain_8class.py'),
                     '--existing', str(EXISTING), '--retrain', str(RETRAIN),
                     '--out', str(MERGED)]).returncode
assert rc == 0, 'merge failed'

In [ ]:
# 3. Step 5 — VALIDATE the merged set. Must print 'VALIDATION PASSED'; raises otherwise.
import subprocess
rc = subprocess.run(['python', str(SCRIPTS / 'validate_retrain_8class.py'),
                     str(MERGED)]).returncode
assert rc == 0, 'merged-set validation FAILED — do NOT train until fixed.'

In [ ]:
# 4. Schema sanity — merged data.yaml must be exactly the frozen 8 classes, bread=7, no counter.
import yaml
cfg = yaml.safe_load((MERGED / 'data.yaml').read_text())
names = cfg['names']
expected = {0:'nut_butter_jar',1:'whole_nuts',2:'hand',3:'cutlery',
            4:'chopping_board',5:'plate',6:'bowl',7:'bread'}
assert names == expected, f'SCHEMA MISMATCH: {names}'
assert 'counter' not in names.values() and names[7] == 'bread'
print('schema OK:', names)

In [ ]:
# 5. Fine-tune FROM the existing checkpoint. Saves under runs/, we copy to the NEW file after.
from ultralytics import YOLO
model = YOLO(str(SRC_CKPT))   # load existing weights (not modified on disk)
results = model.train(
    data=str(MERGED / 'data.yaml'),
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    project='/content/runs_retrain_v2',
    name='finetune',
    exist_ok=True,
    pretrained=True,
    seed=42,
)
print('best weights:', results.save_dir)

In [ ]:
# 6. Save as NEW checkpoint — NEVER overwrite the original.
import shutil
best = Path('/content/runs_retrain_v2/finetune/weights/best.pt')
assert best.exists(), best
assert Path(SRC_CKPT).resolve() != Path(OUT_CKPT).resolve(), 'refusing to overwrite original'
OUT_CKPT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, OUT_CKPT)
print('saved NEW checkpoint ->', OUT_CKPT)
print('original untouched   ->', SRC_CKPT)

In [ ]:
# 7. (optional) Compare new vs old on the merged test split before you trust it.
from ultralytics import YOLO
print('=== OLD checkpoint ==='); YOLO(str(SRC_CKPT)).val(data=str(MERGED/'data.yaml'), split='test')
print('=== NEW checkpoint ==='); YOLO(str(OUT_CKPT)).val(data=str(MERGED/'data.yaml'), split='test')